# Data Cleaning
**This notebook prepares Flatiron Health CSV files for patients with advanced urothelial cancer. Prior to cleaning, the cohort is split into a training and test set (80/20). This notebook focuses on the cleaning of the training set. Each CSV is cleaned using the flatiron_cleaner package. The cleaned dataframes are then merged into a single dataset, which will serve as the input for a gradient boosted survival model.**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorUrothelial
from flatiron_cleaner import merge_dataframes

## Training and test split

In [2]:
df = pd.read_csv('../data/Enhanced_AdvUrothelial.csv')

In [3]:
df.shape

(13129, 13)

In [4]:
df.head(3)

,PatientID,DiagnosisDate,AdvancedDiagnosisDate,PrimarySite,DiseaseGrade,GroupStage,TStage,NStage,MStage,SmokingStatus,Surgery,SurgeryDate,SurgeryType
0,F5AAF96C85477,2021-01-28,2021-05-17,Bladder,High grade (G2/G3/G4),Unknown/not documented,Unknown/not documented,Unknown/not documented,Unknown/not documented,History of smoking,False,NaN,NaN
1,F43136CF07859,2012-06-27,2018-04-02,Renal Pelvis,High grade (G2/G3/G4),Unknown/not documented,Ta,NX,Unknown/not documented,History of smoking,True,2012-07-25,Nephroureterectomy
2,F6FAD468C5AE0,2017-04-06,2018-04-25,Bladder,High grade (G2/G3/G4),Unknown/not documented,Unknown/not documented,Unknown/not documented,M0,No history of smoking,True,2017-08-30,Cystoprostatectomy


In [5]:
df['AdvancedDiagnosisDate'] = pd.to_datetime(df['AdvancedDiagnosisDate'])

In [6]:
df.AdvancedDiagnosisDate.dt.year.value_counts(dropna = False, normalize = True).sort_index()

AdvancedDiagnosisDate
2011    0.042958
2012    0.060858
2013    0.065123
2014    0.073197
2015    0.085841
2016    0.086983
2017    0.092543
2018    0.090258
2019    0.091325
2020    0.091325
2021    0.090411
2022    0.083327
2023    0.045853
Name: proportion, dtype: float64

In [7]:
df['adv_year_cat'] = pd.cut(df['AdvancedDiagnosisDate'].dt.year,
                            bins = [2010.5, 2015.5, 2019.5, 2023.5],
                            labels = ['11-15', '16-19', '20-23'],
                            include_lowest = True)

In [8]:
df.adv_year_cat.value_counts(dropna = False, normalize = True).sort_index()

adv_year_cat
11-15    0.327976
16-19    0.361109
20-23    0.310915
Name: proportion, dtype: float64

In [9]:
train, test = train_test_split(
    df,
    test_size = 0.2,
    stratify = df['adv_year_cat'],  
    random_state = 42
)

In [10]:
train.shape

(10503, 14)

In [11]:
test.shape

(2626, 14)

In [12]:
train.adv_year_cat.value_counts(dropna = False, normalize = True).sort_index()

adv_year_cat
11-15    0.328002
16-19    0.361135
20-23    0.310864
Name: proportion, dtype: float64

In [13]:
test.adv_year_cat.value_counts(dropna = False, normalize = True).sort_index()

adv_year_cat
11-15    0.327875
16-19    0.361005
20-23    0.311120
Name: proportion, dtype: float64

In [14]:
train[['PatientID']].to_csv('../outputs/train_patient_ids.csv', index = False)
test[['PatientID']].to_csv('../outputs/test_patient_ids.csv', index = False)

In [15]:
train_ids = train.PatientID.to_list()

## Data cleaning 

In [16]:
# Initialize class 
processor = DataProcessorUrothelial()

In [17]:
# Index date dataframe
df = train[['PatientID', 'AdvancedDiagnosisDate']]

### Process Enhanced_AdvUrothelial.csv

In [18]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvUrothelial.csv',
                                         patient_ids = train_ids)

2025-04-26 12:20:23,548 - INFO - Successfully read Enhanced_AdvUrothelial.csv file with shape: (13129, 13) and unique PatientIDs: 13129
2025-04-26 12:20:23,548 - INFO - Filtering for 10503 specific PatientIDs
2025-04-26 12:20:23,552 - INFO - Successfully filtered Enhanced_AdvUrothelial.csv file with shape: (10503, 13) and unique PatientIDs: 10503
2025-04-26 12:20:23,570 - INFO - Successfully processed Enhanced_AdvUrothelial.csv file with final shape: (10503, 13) and unique PatientIDs: 10503


In [19]:
enhanced_df['SmokingStatus'] = enhanced_df['SmokingStatus'].map({
    'History of smoking': 1,
    'No history of smoking': 0,
    'Unknown/not documented': 0
})

In [20]:
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)
enhanced_df['days_diagnosis_to_surgery'] = enhanced_df['days_diagnosis_to_surgery'].fillna(0)

In [21]:
enhanced_df['SurgeryType_mod'] = enhanced_df['SurgeryType_mod'].fillna('unknown')

### Process Demographics.csv 

In [22]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = df,
                                                 index_date_column = 'AdvancedDiagnosisDate')

2025-04-26 12:20:23,595 - INFO - Successfully read Demographics.csv file with shape: (13129, 6) and unique PatientIDs: 13129
2025-04-26 12:20:23,606 - WARNING - Found 1 ages outside valid range (18-120)
2025-04-26 12:20:23,613 - INFO - Successfully processed Demographics.csv file with final shape: (10503, 6) and unique PatientIDs: 10503


In [23]:
demographics_df.query('age < 18 or age > 120')

/var/folders/lr/vkkcj_s12115sxc05ly3mshh0000gn/T/ipykernel_40787/2198258583.py:1: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  demographics_df.query('age < 18 or age > 120')


,PatientID,Gender,age,Ethnicity_mod,Race_mod,region
1103,FA857117AA825,M,10,Not Hispanic or Latino,NaN,unknown


In [24]:
demographics_df = demographics_df.query('age >=18')

/var/folders/lr/vkkcj_s12115sxc05ly3mshh0000gn/T/ipykernel_40787/1935262386.py:1: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  demographics_df = demographics_df.query('age >=18')


In [25]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M      7647
F      2853
NaN       2
Name: count, dtype: int64

In [26]:
# Impute missing with male
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

/var/folders/lr/vkkcj_s12115sxc05ly3mshh0000gn/T/ipykernel_40787/3293819633.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)


In [27]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvUrothelialBiomarkers.csv

In [28]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvUrothelialBiomarkers.csv',
                                             index_date_df = df, 
                                             index_date_column = 'AdvancedDiagnosisDate',
                                             days_before = None, 
                                             days_after = 14)

2025-04-26 12:20:23,655 - INFO - Successfully read Enhanced_AdvUrothelialBiomarkers.csv file with shape: (9924, 19) and unique PatientIDs: 4251
2025-04-26 12:20:23,669 - INFO - Successfully merged Enhanced_AdvUrothelialBiomarkers.csv df with index_date_df resulting in shape: (7917, 20) and unique PatientIDs: 3403
2025-04-26 12:20:23,694 - INFO - Successfully processed Enhanced_AdvUrothelialBiomarkers.csv file with final shape: (10503, 4) and unique PatientIDs: 10503


In [29]:
biomarkers_df.PDL1_percent_staining.value_counts(dropna = False)

PDL1_percent_staining
NaN          10481
30% - 39%        6
5% - 9%          3
10% - 19%        3
20% - 29%        2
50% - 59%        2
60% - 69%        2
90% - 99%        2
1%               1
80% - 89%        1
0%               0
< 1%             0
2% - 4%          0
40% - 49%        0
70% - 79%        0
100%             0
Name: count, dtype: int64

In [30]:
def map_pdl1(value):
    if pd.isna(value):  # leave missing as is
        return value
    elif value in ['0%', '< 1%']:
        return '0%'
    else:
        return '>=1%'

biomarkers_df['PDL1_binary'] = biomarkers_df['PDL1_percent_staining'].apply(map_pdl1)

In [31]:
biomarkers_df.PDL1_binary.value_counts(dropna = False)

PDL1_binary
NaN     10481
>=1%       22
Name: count, dtype: int64

In [32]:
biomarkers_df = biomarkers_df.drop(columns = ['PDL1_percent_staining'])

In [33]:
biomarkers_df['FGFR_status'] = biomarkers_df['FGFR_status'].fillna('unknown')
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

### Process ECOG.csv

In [34]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = df,
                                 index_date_column = 'AdvancedDiagnosisDate',
                                 days_before = 90,
                                 days_after = 14,
                                 days_before_further = 180)

2025-04-26 12:20:23,791 - INFO - Successfully read ECOG.csv file with shape: (184794, 4) and unique PatientIDs: 9933
2025-04-26 12:20:23,839 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (146618, 5) and unique PatientIDs: 7924
2025-04-26 12:20:23,913 - INFO - Successfully processed ECOG.csv file with final shape: (10503, 3) and unique PatientIDs: 10503


In [35]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].cat.add_categories('unknown').fillna('unknown')

ecog_df['ecog_index'] = ecog_df["ecog_index"].map({
    0: '0-1',
    1: '0-1',
    2: '2',
    3: '3-4',
    4: '3-4',
    'unknown': 'unknown'
})

ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('category')

In [36]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [37]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = df,
                                     index_date_column = 'AdvancedDiagnosisDate',
                                     weight_days_before = 90,
                                     days_after = 14,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2025-04-26 12:20:27,207 - INFO - Successfully read Vitals.csv file with shape: (3604484, 16) and unique PatientIDs: 13109
2025-04-26 12:20:29,310 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (2897265, 17) and unique PatientIDs: 10488
2025-04-26 12:20:30,324 - INFO - Successfully processed Vitals.csv file with final shape: (10503, 8) and unique PatientIDs: 10503


### Process Lab.csv

In [38]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = df,
                                 index_date_column = 'AdvancedDiagnosisDate',
                                 days_before = 90,
                                 days_after = 14,
                                 summary_lookback = 180)

2025-04-26 12:20:43,669 - INFO - Successfully read Lab.csv file with shape: (9373598, 17) and unique PatientIDs: 12700
2025-04-26 12:20:48,896 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (7485152, 18) and unique PatientIDs: 10160
2025-04-26 12:21:01,404 - INFO - Successfully processed Lab.csv file with final shape: (10503, 76) and unique PatientIDs: 10503


### Process MedicationAdministration.csv

In [39]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = df,
                                               index_date_column = 'AdvancedDiagnosisDate',
                                               days_before = 90,
                                               days_after = 0)

2025-04-26 12:21:02,649 - INFO - Successfully read MedicationAdministration.csv file with shape: (997836, 11) and unique PatientIDs: 10983
2025-04-26 12:21:03,057 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (790064, 12) and unique PatientIDs: 8733
2025-04-26 12:21:03,112 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (10503, 9) and unique PatientIDs: 10503


### Process Diagnosis.csv

In [40]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = df,
                                           index_date_column = 'AdvancedDiagnosisDate',
                                           days_before = None,
                                           days_after = 14)

2025-04-26 12:21:03,519 - INFO - Successfully read Diagnosis.csv file with shape: (625348, 6) and unique PatientIDs: 13129
2025-04-26 12:21:03,679 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (502434, 7) and unique PatientIDs: 10503
2025-04-26 12:21:04,957 - INFO - Successfully processed Diagnosis.csv file with final shape: (10503, 40) and unique PatientIDs: 10503


In [41]:
diagnosis_df['other_gi_met'] = (
    diagnosis_df['adrenal_met'] | diagnosis_df['peritoneum_met'] | diagnosis_df['gi_met']
)

diagnosis_df['other_combined_met'] = (
    diagnosis_df['brain_met'] | diagnosis_df['other_met']
)

diagnosis_df = diagnosis_df.drop(columns = ['adrenal_met', 'peritoneum_met', 'gi_met', 'brain_met', 'other_met'])

## Merge dataframes

In [42]:
final_df = merge_dataframes(demographics_df,
                            enhanced_df,
                            biomarkers_df,
                            ecog_df,
                            vitals_df,
                            labs_df,
                            medications_df,
                            diagnosis_df, 
                            merge_type = 'left')

2025-04-26 12:21:04,973 - INFO - Anticipated number of merges: 7
2025-04-26 12:21:04,973 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 149
2025-04-26 12:21:04,976 - INFO - Dataset 1 shape: (10502, 6), unique PatientIDs: 10502
2025-04-26 12:21:04,978 - INFO - Dataset 2 shape: (10503, 13), unique PatientIDs: 10503
2025-04-26 12:21:04,979 - INFO - Dataset 3 shape: (10503, 4), unique PatientIDs: 10503
2025-04-26 12:21:04,981 - INFO - Dataset 4 shape: (10503, 3), unique PatientIDs: 10503
2025-04-26 12:21:04,983 - INFO - Dataset 5 shape: (10503, 8), unique PatientIDs: 10503
2025-04-26 12:21:04,984 - INFO - Dataset 6 shape: (10503, 76), unique PatientIDs: 10503
2025-04-26 12:21:04,986 - INFO - Dataset 7 shape: (10503, 9), unique PatientIDs: 10503
2025-04-26 12:21:04,988 - INFO - Dataset 8 shape: (10503, 37), unique PatientIDs: 10503
2025-04-26 12:21:05,007 - INFO - After merge 1 shape: (10502, 18), unique PatientIDs 10502
2025

In [43]:
final_df.shape

(10502, 149)

In [44]:
final_df.to_csv('../outputs/mUC_training_df.csv', index = False)

In [45]:
# Save dtypes
final_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/mUC_training_df_dtypes.csv')